# MLP Eve Stage 1, frozen Alice versus XGBoost Stage 2

Small follow-up to [`hybrid_eve_mlp_study.ipynb`](hybrid_eve_mlp_study.ipynb). Alice/Bob use the same 32-bit `CryptCNN`, and Stage 1 uses the same 40,609-parameter ciphertext-only `MLPEve(133, 218)` and loss. Train pairs with seeds 7 and 8, then save both checkpoints. Stage 2 trains 32 independent XGBoost bit classifiers on ciphertext **only**. A fresh MLP and optional random forest see exactly the same frozen-pair examples. Bob alone receives the key for evaluation.

Run in Google Colab with a GPU runtime. Run setup and Stage 1 once; later, reconnect, run setup and definitions, then run Stage 2 with the saved checkpoints. Set `ARTIFACT_DIR` to a mounted Drive folder for persistent checkpoints; `/content` is temporary. The notebook never reads the earlier result files and does not claim any measured result until you run it.

Default per pair: 10,000 Alice/Bob updates, batch 4,096, two MLP Eve updates per Alice/Bob update; Stage 2 has 32,768 train, 8,192 validation, and 16,384 test **messages**, with independently generated random 32-bit keys. The XGBoost budget is 32 × 150 depth-4 histogram trees per pair. Expect Stage 1 to dominate GPU time; XGBoost runs on CPU and may take tens of minutes per pair depending on the Colab machine. Actual times are printed and saved. These smaller Stage 1 settings are a pilot, not an exact reproduction of Week 4's 15,000 steps, batch 16,384 and four Eve updates. Bob's held-out 96% bit-accuracy gate marks a usable pair; weak pairs are saved and reported but skipped for attacks. Review Bob's full-message accuracy as well.

XGBoost's [multi-output support](https://xgboost.readthedocs.io/en/stable/tutorials/multioutput.html) can accept 32 binary columns. Its default `one_output_per_tree` still trains one model per target internally; `multi_output_tree` shares vector-leaf trees but remains experimental and has limited feature support. Explicit per-bit classifiers make the 32 binary tasks and runtime visible, and avoid changing the model class in this pilot. All attackers use ciphertext values only; no key is exposed to their fit or predict calls.


In [ ]:
# Colab setup. Run this cell whenever you reconnect.
%pip -q install 'xgboost>=2.0,<4' scikit-learn

from pathlib import Path
import json, os, time
import numpy as np
import torch
import xgboost as xgb
import sklearn

# Uncomment to persist files in Google Drive, then choose a folder there.
# from google.colab import drive
# drive.mount('/content/drive')
ARTIFACT_DIR = Path('/content/mlp_eve_xgboost_followup')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ATTACK_RUN_NAME = 'pilot1'  # Change for another attack on the saved pairs.
SPLIT_OFFSET = 0           # Change too for new random train/validation/test draws.

PAIR_SEEDS = (7, 8)
PAIR_STEPS = 10_000
STAGE1_BATCH = 4_096
EVE_UPDATES = 2
BOB_MIN_BITS = 0.96
TRAIN_N, VAL_N, TEST_N = 32_768, 8_192, 16_384
TREE_SETTINGS = dict(n_estimators=150, max_depth=4, learning_rate=0.08,
                     subsample=0.9, colsample_bytree=0.9, min_child_weight=5,
                     tree_method='hist', objective='binary:logistic',
                     eval_metric='logloss', n_jobs=2)
RUN_MLP = True
MLP_STEPS, MLP_BATCH = 5_000, 1_024
RUN_RANDOM_FOREST = False  # Optional, slower second tree baseline.
RF_TREES, RF_DEPTH = 100, 12

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime for Stage 1.')
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__,
      'xgboost:', xgb.__version__, 'sklearn:', sklearn.__version__)
print('Artifacts:', ARTIFACT_DIR)


In [ ]:
from contextlib import nullcontext
from torch import nn

BITS = 32

class CryptCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.mix = nn.Linear(64, 192)
        self.conv1 = nn.Conv1d(1, 48, 5, padding='same')
        self.conv2 = nn.Conv1d(48, 96, 4, stride=2, padding=1)
        self.conv3 = nn.Conv1d(96, 48, 2, stride=3)
        self.conv4 = nn.Conv1d(48, 1, 1)

    def forward(self, inputs):
        x = self.mix(inputs).unsqueeze(1)
        x = torch.sigmoid(self.conv1(x))
        x = torch.sigmoid(self.conv2(x))
        x = torch.sigmoid(self.conv3(x))
        return torch.tanh(self.conv4(x).squeeze(1))

class MLPEve(nn.Module):
    def __init__(self, width=133, hidden2=218):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(BITS, width), nn.ReLU(),
                                 nn.Linear(width, hidden2), nn.ReLU(),
                                 nn.Linear(hidden2, BITS), nn.Tanh())

    def forward(self, ciphertext):
        return self.net(ciphertext)

def count(model):
    return sum(p.numel() for p in model.parameters())

assert count(CryptCNN()) == count(MLPEve()) == 40_609

def batch(n, rng):
    message = torch.randint(0, 2, (n, BITS), generator=rng, device=device).float() * 2 - 1
    key = torch.randint(0, 2, (n, BITS), generator=rng, device=device).float() * 2 - 1
    return message, key

def l1_error(prediction, message):
    return ((prediction.float() - message).abs() / 2).sum(dim=1).mean()

def precision():
    if torch.cuda.is_bf16_supported():
        return torch.autocast(device_type='cuda', dtype=torch.bfloat16)
    return nullcontext()

def metrics(predicted_bits, true_bits):
    predicted_bits = np.asarray(predicted_bits, dtype=np.uint8)
    true_bits = np.asarray(true_bits, dtype=np.uint8)
    if predicted_bits.shape != true_bits.shape or true_bits.ndim != 2 or true_bits.shape[1] != BITS:
        raise ValueError('Expected matching arrays of shape (messages, 32)')
    correct = predicted_bits == true_bits
    return {'bit_accuracy': float(correct.mean()),
            'full_message_accuracy': float(correct.all(axis=1).mean()),
            'messages': int(len(true_bits))}

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')

@torch.inference_mode()
def make_split(alice, bob, n, seed, chunk=2048):
    # Separate generator seed for each split; keys and messages are new draws.
    rng = torch.Generator(device=device).manual_seed(seed)
    ciphertext_parts, message_parts, bob_parts = [], [], []
    for start in range(0, n, chunk):
        message, key = batch(min(chunk, n - start), rng)
        with precision():
            ciphertext = alice(torch.cat((message, key), dim=1))
            bob_output = bob(torch.cat((ciphertext, key), dim=1))
        ciphertext_parts.append(ciphertext.float().cpu().numpy())
        message_parts.append((message >= 0).byte().cpu().numpy())
        bob_parts.append((bob_output >= 0).byte().cpu().numpy())
    return (np.concatenate(ciphertext_parts).astype(np.float32),
            np.concatenate(message_parts), np.concatenate(bob_parts))

def pair_paths(seed):
    return (ARTIFACT_DIR / f'pair_seed{seed}.pt',
            ARTIFACT_DIR / f'pair_seed{seed}_stage1.json')

def load_pair(seed):
    checkpoint_path, _ = pair_paths(seed)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Missing {checkpoint_path}; run Stage 1 or restore the checkpoint')
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if checkpoint['pair_seed'] != seed or checkpoint['bits'] != BITS:
        raise ValueError('Checkpoint seed or bit width mismatch')
    alice, bob = CryptCNN().to(device), CryptCNN().to(device)
    alice.load_state_dict(checkpoint['alice'])
    bob.load_state_dict(checkpoint['bob'])
    return alice.eval().requires_grad_(False), bob.eval().requires_grad_(False), checkpoint


## Stage 1: train and save Alice/Bob

Run once. Existing checkpoint files are reused without retraining. A checkpoint contains model weights, architecture and training settings; it is written before Stage 2. To start a new training configuration, use a new artifact folder. Stage 1 uses fresh on-the-fly batches just as the hybrid study did. The privacy term uses the current MLP Eve's reconstruction error. The threshold is checked on a fresh validation split and saved in the Stage 1 record; Stage 2 checks another held-out test split before attacking.


In [ ]:
def train_pair(seed):
    checkpoint_path, record_path = pair_paths(seed)
    if checkpoint_path.exists():
        if not record_path.exists():
            raise FileNotFoundError(f'Missing Stage 1 record for existing {checkpoint_path}')
        record = json.loads(record_path.read_text(encoding='utf-8'))
        expected = {'pair_seed': seed, 'pair_steps': PAIR_STEPS,
                    'stage1_batch': STAGE1_BATCH, 'eve_updates': EVE_UPDATES}
        if any(record.get(k) != v for k, v in expected.items()):
            raise ValueError(f'Existing pair {seed} has different training settings; use a new folder')
        print('Reusing', checkpoint_path)
        return record

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    alice, bob = CryptCNN().to(device), CryptCNN().to(device)
    torch.manual_seed(20_000 + seed)
    eve = MLPEve().to(device)
    ab_opt = torch.optim.Adam([*alice.parameters(), *bob.parameters()], lr=1e-3)
    eve_opt = torch.optim.Adam(eve.parameters(), lr=1e-3)
    ab_rng = torch.Generator(device=device).manual_seed(100_000 + seed)
    eve_rng = torch.Generator(device=device).manual_seed(120_000 + seed)
    start = time.perf_counter()
    for step in range(1, PAIR_STEPS + 1):
        for _ in range(EVE_UPDATES):
            message, key = batch(STAGE1_BATCH, eve_rng)
            with torch.no_grad(), precision():
                ciphertext = alice(torch.cat((message, key), dim=1))
            with precision():
                eve_loss = l1_error(eve(ciphertext), message)
            eve_opt.zero_grad(set_to_none=True)
            eve_loss.backward()
            eve_opt.step()

        message, key = batch(STAGE1_BATCH, ab_rng)
        eve.requires_grad_(False)
        with precision():
            ciphertext = alice(torch.cat((message, key), dim=1))
            bob_loss = l1_error(bob(torch.cat((ciphertext, key), dim=1)), message)
            eve_error = l1_error(eve(ciphertext), message)
            privacy_loss = ((BITS / 2 - eve_error) / (BITS / 2)).square()
            loss = bob_loss + 8.0 * privacy_loss
        ab_opt.zero_grad(set_to_none=True)
        loss.backward()
        ab_opt.step()
        eve.requires_grad_(True)
        if step == 1 or step % 1000 == 0 or step == PAIR_STEPS:
            print(f'pair={seed} step={step}/{PAIR_STEPS} bob_l1={bob_loss.item():.3f} '
                  f'eve_l1={eve_error.item():.3f} elapsed_min={(time.perf_counter()-start)/60:.1f}', flush=True)
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite Stage 1 loss')

    stage1_seconds = time.perf_counter() - start
    alice.eval().requires_grad_(False)
    bob.eval().requires_grad_(False)
    x_val, y_val, bob_val = make_split(alice, bob, VAL_N, 200_000 + seed)
    bob_validation = metrics(bob_val, y_val)
    record = {'pair_seed': seed, 'pair_steps': PAIR_STEPS, 'stage1_batch': STAGE1_BATCH,
              'eve_updates': EVE_UPDATES, 'privacy_weight': 8.0,
              'learning_rate': 1e-3, 'alice_bob_params_each': count(alice),
              'stage1_eve_params': count(eve), 'stage1_seconds': stage1_seconds,
              'bob_validation': bob_validation,
              'bob_validation_pass': bob_validation['bit_accuracy'] >= BOB_MIN_BITS,
              'validation_seed': 200_000 + seed,
              'torch_version': torch.__version__, 'gpu': torch.cuda.get_device_name(0)}
    checkpoint = {'pair_seed': seed, 'bits': BITS, 'architecture': 'CryptCNN',
                  'alice': {k: v.detach().cpu() for k, v in alice.state_dict().items()},
                  'bob': {k: v.detach().cpu() for k, v in bob.state_dict().items()},
                  'stage1_record': record}
    torch.save(checkpoint, checkpoint_path)
    save_json(record_path, record)
    print('Saved', checkpoint_path, 'Bob validation:', bob_validation)
    return record

stage1_records = [train_pair(seed) for seed in PAIR_SEEDS]


## Stage 2: fresh attack data, fixed trees and held-out evaluation

Run this cell after Stage 1, or after loading saved checkpoints in a new Colab session. Every attacker for a given pair fits the **same** 32,768 labeled `(ciphertext, plaintext)` rows. The validation and test splits use separate RNG seeds and are never used for fitting XGBoost or the forest. The MLP uses validation only to choose its best checkpoint. The Stage 1 validation split is separate from all three Stage 2 splits. Tree settings are fixed before test evaluation. Each XGBoost output bit uses an independent binary classifier with `random_state=500000 + pair_seed*32 + bit`; the MLP uses `600000 + pair_seed`, and the optional forest uses `700000 + pair_seed`. Results include training times and software versions. A low Bob full-message score should be discussed even when the bit threshold passes. For a repeat attack, change `ATTACK_RUN_NAME` and `SPLIT_OFFSET` in setup, rerun setup and definitions, then run this cell; it loads the same checkpoints and writes new result files.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

def fit_xgboost(x_train, y_train, x_val, x_test, pair_seed):
    val_pred = np.empty((len(x_val), BITS), dtype=np.uint8)
    test_pred = np.empty((len(x_test), BITS), dtype=np.uint8)
    start = time.perf_counter()
    for bit in range(BITS):
        seed = 500_000 + pair_seed * BITS + bit
        model = xgb.XGBClassifier(**TREE_SETTINGS, random_state=seed)
        model.fit(x_train, y_train[:, bit])
        val_pred[:, bit] = model.predict(x_val).astype(np.uint8)
        test_pred[:, bit] = model.predict(x_test).astype(np.uint8)
        if (bit + 1) % 8 == 0:
            print(f'XGBoost pair={pair_seed}: {bit+1}/32 bits, '
                  f'{(time.perf_counter()-start)/60:.1f} min', flush=True)
    return val_pred, test_pred, time.perf_counter() - start

def fit_mlp(x_train, y_train, x_val, y_val, x_test, pair_seed):
    seed = 600_000 + pair_seed
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    attacker = MLPEve().to(device)
    optimizer = torch.optim.Adam(attacker.parameters(), lr=1e-3)
    x = torch.from_numpy(x_train).to(device)
    y = torch.from_numpy(y_train.astype(np.float32) * 2 - 1).to(device)
    xv = torch.from_numpy(x_val).to(device)
    xt = torch.from_numpy(x_test).to(device)
    generator = torch.Generator(device=device).manual_seed(seed + 1)
    best_loss, best_state, best_step = float('inf'), None, None
    start = time.perf_counter()
    for step in range(1, MLP_STEPS + 1):
        idx = torch.randint(len(x), (MLP_BATCH,), generator=generator, device=device)
        with precision():
            loss = l1_error(attacker(x[idx]), y[idx])
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % 500 == 0 or step == MLP_STEPS:
            attacker.eval()
            with torch.inference_mode(), precision():
                val_output = attacker(xv)
                target = torch.from_numpy(y_val.astype(np.float32) * 2 - 1).to(device)
                val_loss = l1_error(val_output, target).item()
            if val_loss < best_loss:
                best_loss, best_step = val_loss, step
                best_state = {k: v.detach().clone() for k, v in attacker.state_dict().items()}
            attacker.train()
    attacker.load_state_dict(best_state)
    attacker.eval()
    with torch.inference_mode(), precision():
        val_pred = (attacker(xv) >= 0).byte().cpu().numpy()
        test_pred = (attacker(xt) >= 0).byte().cpu().numpy()
    return val_pred, test_pred, time.perf_counter() - start, best_step, best_loss

def attack_pair(seed):
    result_path = ARTIFACT_DIR / f'pair_seed{seed}_stage2_{ATTACK_RUN_NAME}.json'
    if result_path.exists():
        raise FileExistsError(f'{result_path} exists; choose a new artifact folder or move it first')
    alice, bob, checkpoint = load_pair(seed)
    if checkpoint['stage1_record']['bob_validation']['bit_accuracy'] < BOB_MIN_BITS:
        print(f'SKIP pair={seed}: Bob missed {BOB_MIN_BITS:.0%} validation bit threshold')
        return None
    start = time.perf_counter()
    x_train, y_train, _ = make_split(alice, bob, TRAIN_N, 400_000 + SPLIT_OFFSET + seed)
    x_val, y_val, bob_val = make_split(alice, bob, VAL_N, 500_000 + SPLIT_OFFSET + seed)
    x_test, y_test, bob_test = make_split(alice, bob, TEST_N, 600_000 + SPLIT_OFFSET + seed)
    generation_seconds = time.perf_counter() - start
    bob_validation, bob_test_metrics = metrics(bob_val, y_val), metrics(bob_test, y_test)
    result = {'pair_seed': seed, 'attack_run_name': ATTACK_RUN_NAME,
              'sizes': {'train': TRAIN_N, 'validation': VAL_N, 'test': TEST_N},
              'split_seeds': {'train': 400_000 + SPLIT_OFFSET + seed,
                              'validation': 500_000 + SPLIT_OFFSET + seed,
                              'test': 600_000 + SPLIT_OFFSET + seed},
              'bob_stage1_validation': checkpoint['stage1_record']['bob_validation'],
              'bob_validation': bob_validation, 'bob_test': bob_test_metrics,
              'generation_seconds': generation_seconds, 'tree_settings': TREE_SETTINGS,
              'xgboost_version': xgb.__version__, 'sklearn_version': sklearn.__version__,
              'torch_version': torch.__version__, 'attackers': {}}
    if min(bob_validation['bit_accuracy'], bob_test_metrics['bit_accuracy']) < BOB_MIN_BITS:
        result['status'] = 'skipped_bob_below_threshold'
        save_json(result_path, result)
        print('SKIP attacks for pair', seed, 'Bob validation/test:', bob_validation, bob_test_metrics)
        return result
    result['status'] = 'attacked'
    val_pred, test_pred, seconds = fit_xgboost(x_train, y_train, x_val, x_test, seed)
    result['attackers']['xgboost_32_bits'] = {
        'validation': metrics(val_pred, y_val), 'test': metrics(test_pred, y_test),
        'train_seconds': seconds, 'seed_first_bit': 500_000 + seed * BITS}
    save_json(result_path, result)
    if RUN_MLP:
        val_pred, test_pred, seconds, best_step, best_loss = fit_mlp(
            x_train, y_train, x_val, y_val, x_test, seed)
        result['attackers']['fresh_mlp'] = {
            'validation': metrics(val_pred, y_val), 'test': metrics(test_pred, y_test),
            'train_seconds': seconds, 'seed': 600_000 + seed,
            'best_step': best_step, 'best_validation_l1': best_loss,
            'widths': [133, 218]}
        save_json(result_path, result)
    if RUN_RANDOM_FOREST:
        start = time.perf_counter()
        forest = RandomForestClassifier(n_estimators=RF_TREES, max_depth=RF_DEPTH,
                                        min_samples_leaf=5, n_jobs=2,
                                        random_state=700_000 + seed)
        forest.fit(x_train, y_train)
        result['attackers']['random_forest'] = {
            'validation': metrics(forest.predict(x_val), y_val),
            'test': metrics(forest.predict(x_test), y_test),
            'train_seconds': time.perf_counter() - start,
            'seed': 700_000 + seed, 'trees': RF_TREES, 'max_depth': RF_DEPTH,
            'min_samples_leaf': 5}
        save_json(result_path, result)
    print('Saved', result_path)
    del alice, bob
    torch.cuda.empty_cache()
    return result

stage2_results = [attack_pair(seed) for seed in PAIR_SEEDS]


## Read results and download artifacts

Report the per-pair rows, including Bob's full-message accuracy. With only two pair seeds, show the individual results rather than claiming a stable average. Save or download the `.pt` checkpoints and JSON records; rerunning Stage 2 from those checkpoints still generates fresh, deterministic train/validation/test examples from the documented seeds. Change the split seeds for a new independent attack sample. Do not compare with earlier studies without noting the smaller Stage 1 budget and the finite Stage 2 training set.


In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for seed in PAIR_SEEDS:
    path = ARTIFACT_DIR / f'pair_seed{seed}_stage2_{ATTACK_RUN_NAME}.json'
    if not path.exists():
        continue
    record = json.loads(path.read_text(encoding='utf-8'))
    rows.append({'pair_seed': seed, 'model': 'Bob',
                 'bit_accuracy': record['bob_test']['bit_accuracy'],
                 'full_message_accuracy': record['bob_test']['full_message_accuracy'],
                 'train_seconds': json.loads(pair_paths(seed)[1].read_text())['stage1_seconds']})
    for name, attack in record['attackers'].items():
        rows.append({'pair_seed': seed, 'model': name,
                     'bit_accuracy': attack['test']['bit_accuracy'],
                     'full_message_accuracy': attack['test']['full_message_accuracy'],
                     'train_seconds': attack['train_seconds']})
display(pd.DataFrame(rows))
print('Checkpoint and JSON files:', list(ARTIFACT_DIR.iterdir()))

# To download everything from temporary /content storage:
# from google.colab import files
# import shutil
# archive = shutil.make_archive('/content/mlp_eve_xgboost_followup', 'zip', ARTIFACT_DIR)
# files.download(archive)
